# PierPoint Release Desk: Graph and Swarm in one production system

**Stack:** Strands Agents 1.42.0, Amazon Bedrock, Bedrock AgentCore Runtime 1.14.0, Python 3.11+

This notebook builds one working system, not a tour of features. A container is stuck at a terminal and has to be released. Some of that work is a contract you can be audited on. Some of it is a huddle where nobody knows who to ask next. The first kind goes in a Graph. The second kind goes in a Swarm.

**What you already have from earlier sessions**

| Already covered | Used here as a given |
|---|---|
| Bedrock Converse, model IDs, inference profiles | model construction only, no re-teaching |
| Tools and the `@tool` contract | tools appear as read-only lookups |
| `Agent`, structured output | node bodies and boundary contracts |
| `GraphBuilder`, `Swarm`, agents-as-tools | the two primitives being compared |
| Design patterns session | routing, chaining, evaluator-optimizer referenced, not repeated |

**What is new in this notebook**

| New | Why it matters in production |
|---|---|
| A falsifiable rule for Graph vs Swarm | stops the "swarm because it sounds smart" decision |
| Deterministic nodes as first-class graph citizens | invariants stop being prompt instructions |
| Native approval gates with `event.interrupt()` | a real control, not a polite request to the model |
| Idempotent side effects with a replay ledger | retries stop double-releasing containers |
| Hooks as the audit and cost channel | observability without touching business code |
| Concurrency behaviour of `Graph` objects | one line that silently mixes two customers' requests |
| Deploy and invoke on AgentCore Runtime from this notebook | the same code, running behind an HTTP contract |

**How to run this in VS Code**

1. Select the kernel from a venv that has the packages in the next cell.
2. `aws configure` or export `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_REGION=us-east-1`.
3. Run cells in order. Sections 4 and 5 call Bedrock. Section 9 writes files and starts a local server.
4. Mermaid diagrams in markdown cells render on GitHub and with the VS Code extension `bierner.markdown-mermaid`. Without it the fenced block shows as text, which is still readable.

In [1]:
# Install once, then restart the kernel.
# pip install "strands-agents==1.42.0" "bedrock-agentcore==1.14.0" bedrock-agentcore-starter-toolkit boto3 pydantic requests

import importlib.metadata as md

for pkg, expected in [("strands-agents", "1.42.0"), ("bedrock-agentcore", "1.14.0")]:
    try:
        found = md.version(pkg)
        flag = "ok" if found == expected else "version drift, check the API notes in this notebook"
        print(f"{pkg:<32} {found:<12} {flag}")
    except md.PackageNotFoundError:
        print(f"{pkg:<32} MISSING")

# Everything here was verified against the versions above and re-verified on strands-agents 1.50.2:
# Graph, Swarm, PythonNode, interrupts and hooks behave identically on both.
# One API did move. Agent.structured_output(Model, prompt) is deprecated on newer releases.
# The replacement, used throughout this notebook, is:
#     result = agent(prompt, structured_output_model=Model)
#     report = result.structured_output
# It works on 1.42.0 as well, so the code below runs on either.

strands-agents                   1.42.0       ok
bedrock-agentcore                1.17.0       version drift, check the API notes in this notebook


In [2]:
import asyncio
import json
import logging
import os
import re
import socket
import subprocess
import sys
import time
import uuid
from pathlib import Path
from typing import Annotated, Any, AsyncIterator, Callable, Literal

from botocore.config import Config as BotocoreConfig
from pydantic import BaseModel, BeforeValidator, Field, ValidationError

from strands import Agent, ModelRetryStrategy, tool
from strands.agent.agent_result import AgentResult
from strands.models import BedrockModel
from strands.multiagent import GraphBuilder, Swarm
from strands.multiagent.graph import GraphState
from strands.hooks import (
    AfterNodeCallEvent,
    AfterToolCallEvent,
    BeforeNodeCallEvent,
    HookProvider,
    HookRegistry,
)

REGION = os.environ.get("AWS_REGION", "us-east-1")

# Claude on-demand requires the cross-region inference profile prefix. Nova does not.
REASONING_MODEL = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
CHEAP_MODEL = "amazon.nova-lite-v1:0"

# Structured logs on stdout. This is not a style choice: AgentCore Runtime ships
# container stdout to CloudWatch, so stdout IS your production log pipeline.
logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)
_log = logging.getLogger("release_desk")


def jlog(event: str, **fields: Any) -> None:
    """One line of JSON per event. Greppable by correlation_id in CloudWatch."""
    _log.info(json.dumps({"event": event, **fields}, default=str))


# Adaptive botocore retries absorb Bedrock throttling; the Strands retry strategy
# handles model-level retryable errors above it. Two different layers, both needed.
BOTO_CFG = BotocoreConfig(
    retries={"max_attempts": 5, "mode": "adaptive"},
    read_timeout=90,
    connect_timeout=10,
)
RETRY = ModelRetryStrategy(max_attempts=5, initial_delay=4, max_delay=60)


def build_model(model_id: str, temperature: float = 0.2, max_tokens: int = 900) -> BedrockModel:
    """Single place where models are constructed. Set temperature only: Claude 4.x
    rejects temperature and top_p together."""
    return BedrockModel(
        model_id=model_id,
        region_name=REGION,
        temperature=temperature,
        max_tokens=max_tokens,
        boto_client_config=BOTO_CFG,
    )


print(f"region={REGION}")
print(f"reasoning={REASONING_MODEL}")
print(f"cheap={CHEAP_MODEL}")

region=us-east-1
reasoning=us.anthropic.claude-haiku-4-5-20251001-v1:0
cheap=amazon.nova-lite-v1:0


In [3]:
# Preflight. Fail here, not thirty cells later.
for mid in (REASONING_MODEL, CHEAP_MODEL):
    try:
        probe = Agent(model=build_model(mid, temperature=0.0, max_tokens=16),
                      system_prompt="Reply with exactly one word.",
                      callback_handler=None)
        print(f"{mid:<48} -> {str(probe('Say READY')).strip()}")
    except Exception as exc:
        print(f"{mid:<48} -> FAILED {type(exc).__name__}: {str(exc)[:160]}")

# AccessDeniedException      -> model not enabled or IAM policy missing bedrock:InvokeModel
# ValidationException        -> missing "us." inference profile prefix on a Claude model
# ThrottlingException / 424  -> regional capacity, the adaptive retry config above is your first defence

Found credentials in shared credentials file: ~/.aws/credentials
Creating Strands MetricsClient
us.anthropic.claude-haiku-4-5-20251001-v1:0      -> READY
Found credentials in shared credentials file: ~/.aws/credentials
amazon.nova-lite-v1:0                            -> READY


## 1. The scenario

A container cannot leave PierPoint terminal. The customer wants it released. Nobody knows yet why it is stuck.

**Business invariants.** These are the sentences an auditor will read back to you.

| # | Invariant | Consequence if a model decides it instead of code |
|---|---|---|
| I1 | A release is issued at most once per container per request | Duplicate release, two trucks, one box |
| I2 | Any fee waiver above the approval threshold needs a named human approver | Unapproved revenue leakage, no audit trail |
| I3 | Diagnosis is read-only. Nothing that investigates may write | A specialist releases the container mid-investigation |
| I4 | Every outbound customer message is traceable to the findings behind it | Cannot defend the claim response to an insurer |
| I5 | A capped run ends in a reviewable state, never a half-applied one | Fee posted, release not issued, customer already notified |

**The work, stage by stage.** The only column that matters for the Graph and Swarm decision is the third one.

| # | Stage | Enumerable at design time? | Writes? | Owner |
|---|---|---|---|---|
| 1 | Validate request and requesting party | Yes | No | Graph, deterministic node |
| 2 | Find out why the container is blocked | **No.** Depends on what the last check found | No | **Swarm** |
| 3 | Turn findings into a remediation plan | Yes | No | Graph, model node with a typed contract |
| 4 | Approve if exposure crosses the threshold | Yes | No | Graph, interrupt before the node |
| 5 | Apply effects: waive, release, notify | Yes | **Yes** | Graph, deterministic node, idempotent |
| 6 | Write the audit record | Yes | Yes | Graph, deterministic node |

One stage out of six is not enumerable. That single row is the entire case for a swarm, and the other five rows are the entire case against using one for anything else.

```mermaid
flowchart LR
    REQ["Release request"] --> V["1 validate"]
    V --> D["2 diagnose"]
    D --> P["3 plan"]
    P --> A["4 approve"]
    A --> E["5 apply effects"]
    E --> AU["6 audit"]
    subgraph SWARM ["Swarm, read only, bounded"]
      D
    end
    subgraph GRAPH ["Graph, invariants and side effects"]
      V
      P
      A
      E
      AU
    end
```

## 2. The mental model

**Graph is a contract. Swarm is a huddle.**

A contract says what is allowed to happen and in what order. A huddle works out who to ask next. Enterprises need both, in different places, and the failure mode in the field is always the same: someone puts the huddle in charge of the contract.

### Who owns what

| Concern | Graph | Swarm |
|---|---|---|
| Business invariants | Owns them. Conditions on edges are plain Python | Cannot enforce. A system prompt is a request, not a constraint |
| Sequencing | Owns it. The topology *is* the sequence | Emergent. Order changes run to run |
| Approval gates | Owns them. `event.interrupt()` before the guarded node | No gate primitive exists |
| Side effects | Owns them. One node, idempotent, downstream of the gate | Must never write. Ever |
| Provenance | Owns it. `execution_order` plus per-node results | `node_history` says who spoke, not which claim came from where |
| Unknown next specialist | Cannot express it without combinatorial edges | Owns it. `handoff_to_agent` decides at runtime |
| Bounded exploration | Awkward. Needs cycles, revisit rules, caps | Owns it. `max_handoffs` is the natural bound |
| Cost predictability | High. Bounded by topology | Lower. Bounded only by the caps you set |

### The five questions

| # | Question | Yes means |
|---|---|---|
| Q1 | Can you enumerate the legal next steps at design time? | Graph |
| Q2 | Would a wrong order break a business rule or duplicate a side effect? | Graph |
| Q3 | Is the work read-only, with an unknown number of hops? | Swarm |
| Q4 | Must you prove which participant produced a specific claim? | Graph, or typed swarm output plus deterministic assembly |
| Q5 | Would you let a human huddle handle this with no checklist? | Swarm |

```mermaid
flowchart TD
    S["A stage in your workflow"] --> W{"Does it write, or trigger money, or send to a customer?"}
    W -->|Yes| G1["Graph node, downstream of a gate, idempotent"]
    W -->|No| E{"Can you name the next step at design time?"}
    E -->|Yes| G2["Graph node or edge condition"]
    E -->|No| H{"Is the hop count bounded and the work read only?"}
    H -->|Yes| SW["Swarm, capped, typed output at the boundary"]
    H -->|No| STOP["Not an agent problem yet. Narrow the scope first"]
```

### Two rules you can hold someone to in review

1. **If you can name the next step at design time, an LLM must not be the thing that picks it.**
2. **If a step writes, it does not get to decide whether it runs.**

Both rules are testable in code review. That is the point of writing them this way.

### Why not just build one big graph

The blocker can be customs, billing, damage, or equipment. Two can co-occur.

| Design | Paths to encode | Cost of a fifth blocker type |
|---|---|---|
| Fixed graph, one branch per blocker combination | 4 singles plus 6 pairs = 10 | Rises to 15 paths, a code change and a redeploy |
| Swarm of 4 specialists with a handoff tool | 4 nodes, 1 tool | One more node, no topology change |

The swarm is not smarter. It is cheaper to maintain **for this one stage**, because the branch factor lives at runtime instead of in your source tree.

### The four ways teams get this wrong

| Anti-pattern | What it looks like | What actually breaks |
|---|---|---|
| Swarm with write tools | A specialist can issue the release | Two specialists both issue it. No gate, no idempotency, no audit |
| Graph as a diagnosis engine | Fourteen conditional edges guessing the blocker | Every new blocker type is a deploy |
| Prose across the boundary | The execute node parses the swarm's paragraph | Wording drifts, the parse breaks silently, in production |
| Approval as a prompt instruction | "Ask the human before releasing" | The model complies most of the time. Most is not a control |

## 3. Layer one: the diagnostic swarm

Four specialists, one read-only fence. The fence is enforced by the tool set, not by the prompt: none of these tools can write. That is invariant I3, implemented as an absence rather than as a rule.

```mermaid
flowchart TD
    ENTRY["Entry: customs specialist"] --> C["customs_specialist"]
    C -.->|handoff| B["billing_specialist"]
    C -.->|handoff| D["damage_specialist"]
    C -.->|handoff| E["equipment_specialist"]
    B -.->|handoff| D
    B -.->|handoff| E
    D -.->|handoff| B
    E -.->|handoff| B
    C --> T1["customs_status"]
    B --> T2["billing_status"]
    D --> T3["damage_survey"]
    E --> T4["equipment_status"]
    T1 --> RO["Read only. No tool in this layer mutates anything"]
    T2 --> RO
    T3 --> RO
    T4 --> RO
```

The dotted edges are not in your source code. They are what `handoff_to_agent` can do at runtime. That is the difference between a swarm and a graph in one picture.

In [4]:
# --- Read-only systems of record. In production these are API clients. ---
CUSTOMS = {
    "MSCU7391045": {"hold": "documentary", "reason": "commercial invoice value mismatch", "cleared": False},
    "CAIU9083321": {"hold": "none", "reason": "", "cleared": True},
}
BILLING = {
    "MSCU7391045": {"demurrage_days": 6, "accrued_usd": 1800.0, "dispute_open": True},
    "CAIU9083321": {"demurrage_days": 0, "accrued_usd": 0.0, "dispute_open": False},
}
SURVEY = {
    "MSCU7391045": {"survey_done": False, "damage": "none reported"},
    "CAIU9083321": {"survey_done": True, "damage": "none"},
}
EQUIPMENT = {
    "MSCU7391045": {"fault": "none", "reefer_required": False},
    "CAIU9083321": {"fault": "reefer plug bay R04 no power", "reefer_required": True},
}

READ_ONLY_TOOL_NAMES: set[str] = set()


def _lookup(table: dict, container_id: str, label: str) -> str:
    """Shared not-found contract. Never raise, never return empty: the model needs
    a true sentence it can relay."""
    rec = table.get(container_id.strip().upper())
    if rec is None:
        return f"No {label} record exists for container '{container_id}'."
    return json.dumps(rec)


@tool
def customs_status(container_id: str) -> str:
    """Read the customs hold status for one container.

    Args:
        container_id: ISO 6346 number, four letters then seven digits, for example MSCU7391045.
    """
    jlog("tool_call", tool="customs_status", container_id=container_id)
    return _lookup(CUSTOMS, container_id, "customs")


@tool
def billing_status(container_id: str) -> str:
    """Read demurrage, accrued charges and dispute state for one container.

    Args:
        container_id: ISO 6346 number, four letters then seven digits.
    """
    jlog("tool_call", tool="billing_status", container_id=container_id)
    return _lookup(BILLING, container_id, "billing")


@tool
def damage_survey(container_id: str) -> str:
    """Read the damage survey record for one container.

    Args:
        container_id: ISO 6346 number, four letters then seven digits.
    """
    jlog("tool_call", tool="damage_survey", container_id=container_id)
    return _lookup(SURVEY, container_id, "survey")


@tool
def equipment_status(container_id: str) -> str:
    """Read equipment and reefer power faults affecting one container.

    Args:
        container_id: ISO 6346 number, four letters then seven digits.
    """
    jlog("tool_call", tool="equipment_status", container_id=container_id)
    return _lookup(EQUIPMENT, container_id, "equipment")


READ_ONLY_TOOLS = [customs_status, billing_status, damage_survey, equipment_status]
READ_ONLY_TOOL_NAMES = {t.tool_name for t in READ_ONLY_TOOLS}
print("read-only tool set:", sorted(READ_ONLY_TOOL_NAMES))

read-only tool set: ['billing_status', 'customs_status', 'damage_survey', 'equipment_status']


In [5]:
class ReadOnlyFence(HookProvider):
    """Invariant I3 as a runtime assertion. The prompt is not the control; this is.
    Attach to any agent that must not mutate state."""

    def __init__(self, allowed: set[str]) -> None:
        self.allowed = allowed
        self.violations: list[str] = []

    def register_hooks(self, registry: HookRegistry, **kwargs: Any) -> None:
        registry.add_callback(AfterToolCallEvent, self.check)

    def check(self, event: AfterToolCallEvent) -> None:
        name = event.tool_use["name"]
        if name not in self.allowed:
            self.violations.append(name)
            jlog("fence_violation", tool=name)


HANDOFF_BRIEF = (
    "You are one of four PierPoint release-desk specialists: customs_specialist, "
    "billing_specialist, damage_specialist, equipment_specialist.\n"
    "Check your own area with your tool first. Report what you found in two sentences.\n"
    "If your finding points at another area, call handoff_to_agent with a specific question.\n"
    "If nothing else needs checking, state the blocking reason and stop. Do not hand back "
    "to a specialist who already reported.\n"
    "You cannot change anything. You investigate only."
)

SPECIALISTS = {
    "customs_specialist": "Customs holds, documentary discrepancies, inspection state.",
    "billing_specialist": "Demurrage, accrued charges, open billing disputes.",
    "damage_specialist": "Damage surveys, condition disputes.",
    "equipment_specialist": "Reefer power, plug bays, mechanical faults.",
}


def build_diagnostic_swarm() -> tuple[Swarm, ReadOnlyFence]:
    """Fresh instances per request. Agents carry message history, so reusing them
    across requests leaks one customer's case into another's."""
    fence = ReadOnlyFence(READ_ONLY_TOOL_NAMES | {"handoff_to_agent"})
    agents = [
        Agent(
            model=build_model(REASONING_MODEL, temperature=0.2, max_tokens=700),
            name=name,
            system_prompt=f"{HANDOFF_BRIEF}\nYour area: {area}",
            tools=READ_ONLY_TOOLS,
            hooks=[fence],
            retry_strategy=RETRY,
            callback_handler=None,
        )
        for name, area in SPECIALISTS.items()
    ]
    swarm = Swarm(
        agents,
        entry_point=agents[0],
        max_handoffs=6,            # the hop budget, the reason this cannot run away
        max_iterations=8,
        execution_timeout=240.0,
        node_timeout=60.0,
        repetitive_handoff_detection_window=3,
        repetitive_handoff_min_unique_agents=2,   # kills polite ping-pong
    )
    return swarm, fence


def swarm_transcript(result: Any) -> str:
    """MultiAgentResult has no __str__ that yields text. Build the transcript from
    the per-node results, in the order the swarm actually visited them."""
    seen, parts = [], []
    for node in result.node_history:
        if node.node_id in seen:
            continue
        seen.append(node.node_id)
        node_result = result.results.get(node.node_id)
        if node_result is not None:
            parts.append(f"[{node.node_id}]\n{str(node_result.result).strip()}")
    return "\n\n".join(parts)


print("swarm factory ready")

swarm factory ready


In [6]:
CASES = {
    "MSCU7391045": "Shipping line asks why container MSCU7391045 has not been released. Find every blocker.",
    "CAIU9083321": "Cargo owner asks why container CAIU9083321 has not been released. Find every blocker.",
}

diagnoses: dict[str, dict] = {}

for container_id, task in CASES.items():
    swarm, fence = build_diagnostic_swarm()
    started = time.time()
    result = swarm(f"{task}\nContainer id: {container_id}")
    hops = [n.node_id for n in result.node_history]
    diagnoses[container_id] = {
        "status": str(result.status),
        "hops": hops,
        "transcript": swarm_transcript(result),
        "elapsed_s": round(time.time() - started, 1),
        "fence_violations": fence.violations,
    }
    print(f"\n=== {container_id} ===")
    print("status          :", result.status)
    print("path            :", " -> ".join(hops))
    print("unique agents   :", len(set(hops)), "of", len(SPECIALISTS))
    print("hop budget used :", f"{len(hops)}/6")
    print("fence violations:", fence.violations or "none")
    print("elapsed         :", diagnoses[container_id]["elapsed_s"], "s")

Found credentials in shared credentials file: ~/.aws/credentials
Found credentials in shared credentials file: ~/.aws/credentials
Found credentials in shared credentials file: ~/.aws/credentials
Found credentials in shared credentials file: ~/.aws/credentials
{"event": "tool_call", "tool": "customs_status", "container_id": "MSCU7391045"}
{"event": "tool_call", "tool": "billing_status", "container_id": "MSCU7391045"}
{"event": "tool_call", "tool": "damage_survey", "container_id": "MSCU7391045"}
{"event": "tool_call", "tool": "equipment_status", "container_id": "MSCU7391045"}

=== MSCU7391045 ===
status          : Status.COMPLETED
path            : customs_specialist
unique agents   : 1 of 4
hop budget used : 1/6
fence violations: none
elapsed         : 7.1 s
Found credentials in shared credentials file: ~/.aws/credentials
Found credentials in shared credentials file: ~/.aws/credentials
Found credentials in shared credentials file: ~/.aws/credentials
Found credentials in shared credentia

In [7]:
# The transcripts, side by side. Read the paths above before the text below.
for container_id, d in diagnoses.items():
    print(f"\n{'=' * 70}\n{container_id}   path: {' -> '.join(d['hops'])}\n{'=' * 70}")
    print(d["transcript"][:1400])


MSCU7391045   path: customs_specialist
[customs_specialist]
**Summary of All Blockers for MSCU7391045:**

1. **CUSTOMS HOLD (Primary Blocker):** Documentary discrepancy – commercial invoice value mismatch. Hold status: not cleared.

2. **BILLING DISPUTE (Secondary Blocker):** An open dispute exists with 6 days of demurrage accrued ($1,800 USD). This dispute must be resolved before release.

3. **Equipment & Damage:** No faults or damage issues identified.

**Conclusion:** The container cannot be released due to two blockers:
- The customs documentary hold on invoice value mismatch must be resolved and cleared by customs
- The open billing dispute must be settled

Both issues must be addressed before MSCU7391045 can be released.

CAIU9083321   path: customs_specialist -> equipment_specialist
[customs_specialist]
**Customs Finding:** Container CAIU9083321 has no customs hold and is fully cleared. There are no documentary discrepancies or inspection issues in my area.

I've handed off to

### What those two runs actually proved

| Observation | Why it matters |
|---|---|
| The two containers produced different paths through the same four agents | The topology was not in your source code. That is the swarm's only real advantage |
| `node_history` gives you the path, not the provenance | You know who spoke. You do not know which sentence came from which tool result. Insufficient for I4 |
| The fence reported no violations | Invariant I3 held, and you can assert on it in CI rather than hope |
| Hop count sat below the cap | If p95 hops equals `max_handoffs`, your caps are producing the answer, not the diagnosis |

### When the swarm is the wrong choice here

- If the blocker were always customs, this is a single agent with one tool and the swarm is pure overhead.
- If ops needed the path to be identical every time for compliance reasons, a graph with four conditional edges is correct and the swarm is disqualified.
- If any specialist needed to write, the swarm is disqualified immediately. There is no partial version of I3.

### The boundary: prose in, types out

The swarm produced paragraphs. Paragraphs are fine for a human reading a case. They are not an input to a system that moves money.

```mermaid
flowchart LR
    SW["Swarm transcript, prose, variable shape"] --> X{"Boundary"}
    X --> T["DiagnosisReport, typed, validated"]
    T --> G["Graph consumes only this"]
    X -.->|never| G
```

Three reasons the type is not optional:

1. A parse of prose fails silently. A Pydantic validation error fails loudly, in the run that caused it.
2. The typed object is what you store for audit. A paragraph is not evidence, it is a recollection.
3. It is the seam where you can swap the swarm for a single agent, or for a rules engine, without touching the graph.

Cost of the boundary: one extra cheap model call per request. Cheaper than one wrong release.

### The seven ways a typed boundary breaks in production

Every row below has taken down a real run. The next cell handles all seven, and the handling is the interesting part: **coerce where the model is merely sloppy, fail closed where money or safety depends on the field.**

| # | What the model sends | Naive contract does | This notebook does |
|---|---|---|---|
| 1 | `"unresolved": null` for an empty list | raises: `default_factory` fires only when the key is **absent**, not when it is explicitly null | coerce null to `[]` |
| 2 | a 340-character `detail` against `max_length=300` | raises | clip to the limit and log it. A long sentence is not a business failure |
| 3 | `"primary_blocker": "customs hold"` against a `Literal` | raises | normalise to the nearest allowed value, log every normalisation |
| 4 | `"blocking": "yes"` instead of a bool | raises | coerce, and treat anything unreadable as **blocking**, which is the safe direction |
| 5 | `"financial_exposure_usd": "about 1800 USD"` | raises, or worse, coerces to something wrong | parse the number out, and if it cannot be parsed, return **unknown** |
| 6 | omits `financial_exposure_usd` entirely | defaults to `0.0` and **silently passes under the approval threshold** | unknown exposure forces approval. Fail closed on the field the gate depends on |
| 7 | returns no structured output at all | raises deep in a node | one repair attempt with the error fed back, then a model escalation, then fail loudly |

Row 6 is the one worth arguing about in a design review. A missing number that defaults to zero is how an unapproved waiver gets issued, and no test catches it because nothing threw.

**The design goal for a boundary model: it should be unable to raise.** Every field has a documented fallback, every fallback is logged, and the log line is a metric. A rising normalisation rate means the brief is drifting, which you want to know before the output does.

In [8]:
# --- Coercion layer. Shared by every typed boundary in this notebook. ---

def _null_to_list(value: Any) -> Any:
    """Row 1: null where the model meant an empty list."""
    return [] if value is None else value


def _clip(limit: int) -> Callable[[Any], Any]:
    """Row 2: truncate instead of rejecting. Log it, because a rising clip rate means
    the brief asked for less than the model wants to say."""
    def clip(value: Any) -> Any:
        if isinstance(value, str) and len(value) > limit:
            jlog("field_clipped", limit=limit, original_length=len(value))
            return value[:limit]
        return value
    return clip


def _to_choice(allowed: tuple[str, ...], fallback: str) -> Callable[[Any], Any]:
    """Rows 3: normalise a label the model got slightly wrong, rather than losing the run
    over a word. Every normalisation is logged: the rate is the signal."""
    def choose(value: Any) -> Any:
        if isinstance(value, str):
            candidate = value.strip().lower().replace(" ", "_").replace("-", "_")
            if candidate in allowed:
                return candidate
            for option in allowed:
                if option in candidate:
                    jlog("field_normalised", got=value[:60], used=option)
                    return option
        jlog("field_normalised", got=str(value)[:60], used=fallback)
        return fallback
    return choose


def _to_bool(value: Any) -> Any:
    """Row 4: unreadable means blocking. Wrong in the safe direction costs one extra
    remediation step; wrong in the other direction releases a container that should not move."""
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        text = value.strip().lower()
        if text in {"true", "yes", "y", "1", "blocking"}:
            return True
        if text in {"false", "no", "n", "0", "clear", "not_blocking"}:
            return False
    jlog("bool_unreadable", got=str(value)[:40], used=True)
    return True


def _to_money(value: Any) -> Any:
    """Rows 5 and 6: parse a number out of whatever arrived. Return None for unknown,
    and never invent a zero. None is what makes the approval gate fail closed.
    A negative exposure is also treated as unknown: it would slip under the threshold."""
    if value is None:
        return None
    if isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return float(value) if value >= 0 else _unknown_money(value)
    if isinstance(value, str):
        match = re.search(r"-?\d+(?:\.\d+)?", value.replace(",", ""))
        if match:
            parsed = float(match.group())
            return parsed if parsed >= 0 else _unknown_money(value)
    return _unknown_money(value)


def _unknown_money(value: Any) -> None:
    jlog("exposure_unreadable", got=str(value)[:60], used="unknown, approval forced")
    return None


print("coercion layer ready:", [f.__name__ for f in (_null_to_list, _clip, _to_choice, _to_bool, _to_money)])

coercion layer ready: ['_null_to_list', '_clip', '_to_choice', '_to_bool', '_to_money']


In [9]:
AREAS = ("customs", "billing", "damage", "equipment")


class Finding(BaseModel):
    area: Annotated[Literal["customs", "billing", "damage", "equipment"],
                    BeforeValidator(_to_choice(AREAS, "customs"))] = "customs"
    blocking: Annotated[bool, BeforeValidator(_to_bool)] = True
    detail: Annotated[str, BeforeValidator(_clip(300))] = ""


class DiagnosisReport(BaseModel):
    """Designed so it cannot raise. Every field has a fallback, every fallback logs."""

    container_id: str = ""
    findings: Annotated[list[Finding], BeforeValidator(_null_to_list)] = Field(default_factory=list)
    primary_blocker: Annotated[Literal["customs", "billing", "damage", "equipment", "none"],
                               BeforeValidator(_to_choice(AREAS + ("none",), "none"))] = "none"
    unresolved: Annotated[list[str], BeforeValidator(_null_to_list)] = Field(default_factory=list)


CONTRACT_BRIEF = (
    "Convert a release-desk investigation transcript into the required structure. "
    "Include one finding per area that was actually checked. Set blocking=true only "
    "where the transcript says the container cannot move. Never invent an area that "
    "was not checked. Put anything the specialists left open into unresolved. "
    "Use an empty list where a list is empty, never null."
)


def build_contract_agent(model_id: str = CHEAP_MODEL) -> Agent:
    """Fresh agent per extraction. Passing structured_output_model on the invocation adds
    the prompt to conversation history, so a reused agent would carry the previous
    container's transcript into the next one. Same build-per-request rule as the graph."""
    return Agent(
        model=build_model(model_id, temperature=0.0, max_tokens=900),
        name="contract",
        system_prompt=CONTRACT_BRIEF,
        structured_output_model=DiagnosisReport,
        retry_strategy=RETRY,
        callback_handler=None,
    )


def extract_report(container_id: str, transcript: str) -> DiagnosisReport:
    """Row 7: one repair attempt, one model escalation, then fail loudly. A boundary that
    quietly returns a half-parsed object is worse than no boundary at all."""
    prompt = f"Container id: {container_id}\n\nTranscript:\n{transcript}"
    last_error: str | None = None

    for tier, model_id in (("cheap", CHEAP_MODEL), ("reasoning", REASONING_MODEL)):
        ask = prompt if last_error is None else (
            f"{prompt}\n\nYour previous answer failed validation:\n{last_error}\n"
            "Return every required field. Use [] for empty lists, never null."
        )
        try:
            result = build_contract_agent(model_id)(ask, structured_output_model=DiagnosisReport)
            if result.structured_output is None:
                raise ValueError("model returned no structured output")
            jlog("contract_extracted", container_id=container_id, tier=tier)
            return result.structured_output
        except (ValidationError, ValueError) as exc:
            last_error = str(exc)[:400]
            jlog("contract_repair", container_id=container_id, tier=tier, error=last_error[:160])

    raise RuntimeError(f"typed boundary failed for {container_id}: {last_error}")


reports: dict[str, DiagnosisReport] = {}

for container_id, d in diagnoses.items():
    report = extract_report(container_id, d["transcript"])
    reports[container_id] = report
    print(f"\n=== {container_id} ===")
    print("primary_blocker:", report.primary_blocker)
    for f in report.findings:
        print(f"  {'BLOCKING' if f.blocking else 'clear   '} {f.area:<10} {f.detail[:80]}")
    print("unresolved:", report.unresolved or "none")

Found credentials in shared credentials file: ~/.aws/credentials
{"event": "contract_extracted", "container_id": "MSCU7391045", "tier": "cheap"}

=== MSCU7391045 ===
primary_blocker: customs
  BLOCKING customs    Documentary discrepancy – commercial invoice value mismatch. Hold status: not cl
  BLOCKING billing    An open dispute exists with 6 days of demurrage accrued ($1,800 USD). This dispu
unresolved: none
Found credentials in shared credentials file: ~/.aws/credentials
{"event": "contract_extracted", "container_id": "CAIU9083321", "tier": "cheap"}

=== CAIU9083321 ===
primary_blocker: none
  clear    customs    Container CAIU9083321 has no customs hold and is fully cleared. There are no doc
  clear    billing    No demurrage, no accrued charges, no disputes.
  clear    damage     Survey complete, no damage found.
  BLOCKING equipment  Reefer plug bay R04 is not providing power to the container.
unresolved: none


## 4. Layer two: the graph

One line to hold on to:

**Conditions are code. Nodes are models. Every invariant lives on an edge or in a deterministic node.**

`GraphBuilder.add_edge(..., condition=fn)` takes a plain Python callable that receives `GraphState`. That callable is where your business rules belong: it cannot hallucinate, it is unit testable, and it appears in a diff.

`AgentBase` in Strands is a runtime-checkable Protocol with three methods. Anything satisfying it can be a node, which means a pure Python step can be a first-class graph citizen with the same audit trail, timings and hooks as a model node, at zero token cost.

In [10]:
class PythonNode:
    """A deterministic graph node. No model, no tokens, fully unit testable.

    Satisfies the AgentBase protocol: invoke_async, __call__, stream_async.
    Returns a real AgentResult so downstream conditions, str() and the graph's own
    bookkeeping treat it exactly like a model node."""

    def __init__(self, name: str, fn: Callable[[str, dict], dict]) -> None:
        self.name = name
        self._fn = fn

    @staticmethod
    def _wrap(payload: dict) -> AgentResult:
        return AgentResult(
            stop_reason="end_turn",
            message={"role": "assistant", "content": [{"text": json.dumps(payload)}]},
            metrics=None,
            state={},
        )

    async def invoke_async(self, prompt: Any = None, **kwargs: Any) -> AgentResult:
        text = "".join(b.get("text", "") for b in (prompt or []) if isinstance(b, dict))
        payload = self._fn(text, kwargs.get("invocation_state") or {})
        jlog("python_node", node=self.name, payload=payload)
        return self._wrap(payload)

    def __call__(self, prompt: Any = None, **kwargs: Any) -> AgentResult:
        return asyncio.run(self.invoke_async(prompt, **kwargs))

    async def stream_async(self, prompt: Any = None, **kwargs: Any) -> AsyncIterator[Any]:
        yield {"result": await self.invoke_async(prompt, **kwargs)}


def node_json(holder: Any, node_id: str) -> dict:
    """Read a node's output as a dict. Accepts a GraphState (mid-run, from a hook or an
    edge condition) or a GraphResult (after the run): both expose .results.

    AgentResult.__str__ appends a newline, so strip before parsing. A parse failure must
    never be mistaken for a business outcome, so it is logged and flagged."""
    node_result = holder.results.get(node_id)
    if node_result is None:
        return {}
    raw = str(node_result.result).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start, end = raw.find("{"), raw.rfind("}")
        if start >= 0 and end > start:
            try:
                return json.loads(raw[start : end + 1])
            except json.JSONDecodeError:
                pass
        jlog("node_parse_failed", node=node_id, raw=raw[:200])
        return {"_parse_failed": True}


print("PythonNode ready, isinstance check:", hasattr(PythonNode("x", lambda t, s: {}), "stream_async"))

PythonNode ready, isinstance check: True


In [11]:
# --- Policy constants. These belong in config, not in a prompt. ---
APPROVAL_THRESHOLD_USD = 500.0
AUTHORIZED_PARTIES = {"maersk-ops", "cma-cgm-ops", "pierpoint-internal"}
CONTAINER_PATTERN = re.compile(r"^[A-Z]{4}\d{7}$")

# --- Effect ledger. In production: a DynamoDB table with the key as the partition key. ---
EFFECT_LEDGER: dict[str, dict] = {}


def intake_fn(_text: str, state: dict) -> dict:
    """Invariant gate. Runs before any token is spent."""
    container_id = str(state.get("container_id", "")).strip().upper()
    party = str(state.get("party", "")).strip().lower()
    problems = []
    if not CONTAINER_PATTERN.match(container_id):
        problems.append("container_id is not ISO 6346 shape")
    if party not in AUTHORIZED_PARTIES:
        problems.append(f"party '{party}' is not authorized to request a release")
    return {"ok": not problems, "problems": problems, "container_id": container_id, "party": party}


def approval_required(exposure_usd: float | None, threshold_usd: float) -> bool:
    """The fail-closed rule, as a plain function so it can be tested without a graph,
    a model or a network. Unknown exposure requires approval: a missing number must never
    behave like a zero."""
    return exposure_usd is None or exposure_usd > threshold_usd


def approval_fn(_text: str, state: dict) -> dict:
    """Records the approval outcome that the hook already decided. This node does not
    decide anything: it is the audit surface for the decision."""
    decision = state.get("approval_outcome")
    if decision is None:
        return {"approved": False, "by": "missing", "reason": "no approval decision present"}
    return decision


def apply_effects_fn(_text: str, state: dict) -> dict:
    """Invariant I1. The idempotency key makes a retry a no-op instead of a second release.

    Never raises. A raise inside a side-effect node skips the audit node, which leaves a
    run with no record of whether anything was applied. An error payload keeps the audit."""
    container_id = state.get("container_id")
    correlation_id = state.get("correlation_id")
    if not container_id or not correlation_id:
        jlog("apply_effects_missing_context", container_id=container_id, correlation_id=correlation_id)
        return {"status": "error", "reason": "missing container_id or correlation_id", "effects": []}

    plan = state.get("plan") or {}
    key = f"{correlation_id}:release:{container_id}"
    if key in EFFECT_LEDGER:
        return {"status": "already_applied", "key": key, "effects": EFFECT_LEDGER[key]["effects"]}
    effects = []
    for step in plan.get("steps") or []:
        effects.append({"action": step.get("action"), "target": step.get("target"), "result": "applied"})
    effects.append({"action": "release_container", "target": container_id, "result": "applied"})
    EFFECT_LEDGER[key] = {"effects": effects, "exposure_usd": plan.get("financial_exposure_usd", 0.0)}
    return {"status": "applied", "key": key, "effects": effects}


def rejected_fn(_text: str, state: dict) -> dict:
    """Terminal path. No side effect on the container, one notification, one audit row."""
    return {
        "outcome": "not_released",
        "container_id": state.get("container_id"),
        "reason": state.get("rejection_reason", "approval not granted"),
        "customer_notified": True,
    }


def audit_fn(_text: str, state: dict) -> dict:
    return {
        "audit_written": True,
        "correlation_id": state.get("correlation_id"),
        "container_id": state.get("container_id"),
    }


print("deterministic node bodies ready")

deterministic node bodies ready


In [12]:
ACTIONS = (
    "clear_customs_hold",
    "waive_demurrage",
    "post_charge",
    "dispatch_equipment_repair",
    "request_survey",
    "notify_party",
)


class PlanStep(BaseModel):
    action: Annotated[Literal["clear_customs_hold", "waive_demurrage", "post_charge",
                              "dispatch_equipment_repair", "request_survey", "notify_party"],
                      BeforeValidator(_to_choice(ACTIONS, "notify_party"))] = "notify_party"
    target: str = ""
    rationale: Annotated[str, BeforeValidator(_clip(200))] = ""


class RemediationPlan(BaseModel):
    """Same rule as DiagnosisReport: it cannot raise. The one field that is allowed to be
    unknown is the money field, and unknown there means the approval gate fires."""

    container_id: str = ""
    steps: Annotated[list[PlanStep], BeforeValidator(_null_to_list)] = Field(default_factory=list)
    financial_exposure_usd: Annotated[float | None, BeforeValidator(_to_money)] = None
    model_thinks_approval_needed: Annotated[bool, BeforeValidator(_to_bool)] = False
    customer_message: Annotated[str, BeforeValidator(_clip(600))] = ""


PLANNER_BRIEF = (
    "You are the PierPoint release desk planner. You receive a typed diagnosis and produce "
    "the remediation plan.\n"
    "You receive a validated DiagnosisReport as JSON, not a transcript.\n"
    "Rules: one step per blocking finding, no steps for areas that are clear. "
    "financial_exposure_usd is the total value PierPoint gives up or defers under this plan, "
    "and it must always be a number, using 0 when nothing is given up. "
    "Never admit liability in customer_message. Under 120 words."
)


def build_planner() -> Agent:
    """A model node with structured_output_model set: the node's output is typed at the
    edge, so downstream conditions read fields, not sentences."""
    return Agent(
        model=build_model(REASONING_MODEL, temperature=0.2, max_tokens=1200),
        name="planner",
        system_prompt=PLANNER_BRIEF,
        structured_output_model=RemediationPlan,
        retry_strategy=RETRY,
        callback_handler=None,
    )


print("planner factory ready")

planner factory ready


In [13]:
class ReleaseControls(HookProvider):
    """Every production control that does not belong in business logic.

    Approval gate  : interrupts before the approval node when exposure crosses policy
    Policy override: recomputes the approval decision in code, ignoring the model's opinion
    Audit          : node-level timings and ordering
    Divergence     : records where the model's judgement disagreed with policy
    """

    def __init__(self, threshold_usd: float = APPROVAL_THRESHOLD_USD) -> None:
        self.threshold = threshold_usd
        self.audit: list[dict] = []
        self._started: dict[str, float] = {}
        self.divergences: list[dict] = []

    def register_hooks(self, registry: HookRegistry, **kwargs: Any) -> None:
        registry.add_callback(BeforeNodeCallEvent, self.before_node)
        registry.add_callback(AfterNodeCallEvent, self.after_node)

    def before_node(self, event: BeforeNodeCallEvent) -> None:
        self._started[event.node_id] = time.time()

        if event.node_id == "rejected" and "rejection_reason" not in event.invocation_state:
            problems = node_json(event.source.state, "intake").get("problems") or []
            if problems:
                event.invocation_state["rejection_reason"] = "; ".join(problems)

        if event.node_id != "approval":
            return

        plan = node_json(event.source.state, "plan")
        # Hooks are the glue: they read graph state and write into invocation_state,
        # which is how a downstream node receives data from a non-adjacent node.
        event.invocation_state["plan"] = plan

        # Re-coerce here as well. The gate must not trust that the value reaching it
        # went through the contract, because a parse failure upstream returns a raw dict.
        exposure = _to_money(plan.get("financial_exposure_usd"))
        model_opinion = _to_bool(plan.get("model_thinks_approval_needed", False))
        policy_requires = approval_required(exposure, self.threshold)

        if model_opinion != policy_requires:
            # Not an error. A metric. Persistent divergence means the brief is wrong.
            self.divergences.append({"exposure": exposure, "model": model_opinion, "policy": policy_requires})
            jlog("policy_divergence", exposure_usd=exposure, model=model_opinion, policy=policy_requires)

        if not policy_requires:
            event.invocation_state["approval_outcome"] = {
                "approved": True, "by": "policy", "exposure_usd": exposure,
                "rule": f"exposure {exposure} <= threshold {self.threshold}",
            }
            self.audit.append({"node": "approval", "auto_approved": True, "exposure": exposure})
            return

        if exposure is None:
            jlog("exposure_unknown_approval_forced", node=event.node_id)

        # Invariant I2. Execution stops here and returns to the caller.
        decision = event.interrupt(
            "release_approval",
            reason={
                "container_id": plan.get("container_id"),
                "exposure_usd": exposure,
                "exposure_known": exposure is not None,
                "threshold_usd": self.threshold,
                "steps": [s.get("action") for s in plan.get("steps") or []],
            },
        )
        event.invocation_state["approval_outcome"] = {
            "approved": bool(decision.get("approved")),
            "by": decision.get("approver", "unknown"),
            "exposure_usd": exposure,
            "rule": "human approval",
        }
        if not decision.get("approved"):
            event.invocation_state["rejection_reason"] = decision.get("note", "approval denied")
        self.audit.append({"node": "approval", "auto_approved": False, "decision": decision})

    def after_node(self, event: AfterNodeCallEvent) -> None:
        started = self._started.pop(event.node_id, None)
        entry = {"node": event.node_id, "ms": round((time.time() - started) * 1000) if started else None}
        self.audit.append(entry)
        jlog("node_complete", **entry)


print("controls ready")

controls ready


In [14]:
def request_approved(state: GraphState) -> bool:
    return bool(node_json(state, "approval").get("approved"))


def request_denied(state: GraphState) -> bool:
    return "approval" in state.results and not node_json(state, "approval").get("approved")


def intake_ok(state: GraphState) -> bool:
    return bool(node_json(state, "intake").get("ok"))


def intake_failed(state: GraphState) -> bool:
    return "intake" in state.results and not node_json(state, "intake").get("ok")


def build_release_graph(controls: ReleaseControls, diagnostic_swarm: Swarm):
    """A graph per request. Not a module-level singleton. See the concurrency cell below
    for the failure this avoids."""
    builder = GraphBuilder()

    builder.add_node(PythonNode("intake", intake_fn), "intake")
    builder.add_node(diagnostic_swarm, "diagnose")          # a Swarm is a legal graph node
    # prose in, types out. Cheap model by default: this is extraction, not reasoning.
    # If the contract node is where your runs fail, this is the one line to escalate.
    builder.add_node(build_contract_agent(CHEAP_MODEL), "contract")
    builder.add_node(build_planner(), "plan")
    builder.add_node(PythonNode("approval", approval_fn), "approval")
    builder.add_node(PythonNode("apply_effects", apply_effects_fn), "apply_effects")
    builder.add_node(PythonNode("rejected", rejected_fn), "rejected")
    builder.add_node(PythonNode("audit", audit_fn), "audit")

    builder.add_edge("intake", "diagnose", condition=intake_ok)
    builder.add_edge("intake", "rejected", condition=intake_failed)
    builder.add_edge("diagnose", "contract")
    builder.add_edge("contract", "plan")
    builder.add_edge("plan", "approval")
    builder.add_edge("approval", "apply_effects", condition=request_approved)
    builder.add_edge("approval", "rejected", condition=request_denied)
    builder.add_edge("apply_effects", "audit")
    builder.add_edge("rejected", "audit")

    builder.set_entry_point("intake")
    builder.set_hook_providers([controls])
    builder.set_max_node_executions(16)     # bounded blast radius
    builder.set_node_timeout(120.0)
    builder.set_execution_timeout(600.0)
    return builder.build()


print("graph factory ready")

graph factory ready


### The assembled system

```mermaid
flowchart TD
    START["Release request"] --> IN["intake, python node"]
    IN -->|"condition: intake ok"| DG["diagnose, SWARM nested as one node"]
    IN -->|"condition: intake failed"| RJ["rejected, python node"]
    DG --> CT["contract, model node, prose in types out"]
    CT --> PL["plan, model node, typed output"]
    PL --> AP["approval, python node"]
    AP -->|"condition: approved"| AE["apply_effects, python node, idempotent"]
    AP -->|"condition: denied"| RJ
    AE --> AU["audit, python node"]
    RJ --> AU
    GATE["ReleaseControls hook: interrupts before approval when exposure crosses threshold"] -.-> AP
```

Read the diagram against the invariants:

| Invariant | Where it is enforced |
|---|---|
| I1 one release per request | `apply_effects` idempotency key, not a prompt |
| I2 human approver above threshold | hook interrupt before `approval`, threshold in config |
| I3 diagnosis is read-only | tool set of the swarm plus `ReadOnlyFence` |
| I4 traceable messages | the `contract` node stores a validated `DiagnosisReport`, `plan` output is typed, `execution_order` gives the path |
| I5 capped runs are reviewable | caps end the run before `apply_effects`, so nothing is half applied |

### Why denial is an edge and not an exception

`BeforeNodeCallEvent` also exposes `cancel_node`. Setting it raises `RuntimeError` out of the graph call, which is correct for a kill switch and wrong for a business outcome: you lose the audit node, the customer notification and the result object.

**Rule: `cancel_node` is for aborting a run. A denied approval is a destination, so it gets an edge.**

In [15]:
def run_release(container_id: str, party: str, correlation_id: str | None = None,
                approval_response: Any = None) -> dict:
    """One request, one graph, one correlation id. Returns everything needed to audit it."""
    correlation_id = correlation_id or f"req-{uuid.uuid4().hex[:8]}"
    controls = ReleaseControls()
    swarm, fence = build_diagnostic_swarm()
    graph = build_release_graph(controls, swarm)

    state = {"correlation_id": correlation_id, "container_id": container_id, "party": party}
    task = (
        f"Release request for container {container_id} from {party}. "
        f"Diagnose every blocker, then plan the remediation."
    )
    jlog("request_start", correlation_id=correlation_id, container_id=container_id, party=party)

    result = graph(task, invocation_state=state)

    pending = None
    if result.status.name == "INTERRUPTED" and result.interrupts:
        pending = result.interrupts[0]
        jlog("approval_required", correlation_id=correlation_id, reason=pending.reason)
        if approval_response is not None:
            result = graph(
                [{"interruptResponse": {"interruptId": pending.id, "response": approval_response}}],
                invocation_state=state,
            )
            pending = None

    return {
        "correlation_id": correlation_id,
        "graph": graph,
        "result": result,
        "controls": controls,
        "fence": fence,
        "pending_approval": pending,
    }


def show(run: dict) -> None:
    r = run["result"]
    print("correlation_id :", run["correlation_id"])
    print("status         :", r.status)
    print("path           :", " -> ".join(n.node_id for n in r.execution_order))
    print("node executions:", r.execution_count, "| distinct nodes:", r.completed_nodes)
    print("tokens         :", r.accumulated_usage)
    if run["pending_approval"] is not None:
        print("PENDING APPROVAL:", json.dumps(run["pending_approval"].reason, indent=2))
    for node_id in ("contract", "plan", "approval", "apply_effects", "rejected"):
        if node_id in r.results:
            print(f"  {node_id:<14}", str(r.results[node_id].result).strip()[:220])


print("runner ready")

runner ready


In [16]:
# RUN 1. CAIU9083321: equipment fault, no money on the line. Policy auto-approves.
run1 = run_release("CAIU9083321", "cma-cgm-ops")
show(run1)
print("\nledger:", json.dumps(EFFECT_LEDGER, indent=2)[:600])

Found credentials in shared credentials file: ~/.aws/credentials
Found credentials in shared credentials file: ~/.aws/credentials
Found credentials in shared credentials file: ~/.aws/credentials
Found credentials in shared credentials file: ~/.aws/credentials
Found credentials in shared credentials file: ~/.aws/credentials
Found credentials in shared credentials file: ~/.aws/credentials
{"event": "request_start", "correlation_id": "req-297094d7", "container_id": "CAIU9083321", "party": "cma-cgm-ops"}
{"event": "python_node", "node": "intake", "payload": {"ok": true, "problems": [], "container_id": "CAIU9083321", "party": "cma-cgm-ops"}}
{"event": "node_complete", "node": "intake", "ms": 3}
{"event": "tool_call", "tool": "customs_status", "container_id": "CAIU9083321"}
{"event": "tool_call", "tool": "equipment_status", "container_id": "CAIU9083321"}
{"event": "tool_call", "tool": "billing_status", "container_id": "CAIU9083321"}
{"event": "tool_call", "tool": "damage_survey", "container_

In [17]:
# RUN 2. MSCU7391045: customs hold plus 1800 USD demurrage. Exposure crosses the
# threshold, so the graph stops before the approval node and hands control back.
run2 = run_release("MSCU7391045", "maersk-ops")
show(run2)
print("\ninterrupted_nodes:", run2["result"].interrupted_nodes)
print("effects applied so far:", len(EFFECT_LEDGER), "(unchanged: nothing was written)")

Found credentials in shared credentials file: ~/.aws/credentials
Found credentials in shared credentials file: ~/.aws/credentials
Found credentials in shared credentials file: ~/.aws/credentials
Found credentials in shared credentials file: ~/.aws/credentials
Found credentials in shared credentials file: ~/.aws/credentials
Found credentials in shared credentials file: ~/.aws/credentials
{"event": "request_start", "correlation_id": "req-69f22a9f", "container_id": "MSCU7391045", "party": "maersk-ops"}
{"event": "python_node", "node": "intake", "payload": {"ok": true, "problems": [], "container_id": "MSCU7391045", "party": "maersk-ops"}}
{"event": "node_complete", "node": "intake", "ms": 2}
{"event": "tool_call", "tool": "customs_status", "container_id": "MSCU7391045"}
{"event": "tool_call", "tool": "billing_status", "container_id": "MSCU7391045"}
{"event": "tool_call", "tool": "customs_status", "container_id": "MSCU7391045"}
{"event": "node_complete", "node": "diagnose", "ms": 21262}
{"e

In [ ]:
# RUN 2 continued. A human approves. Resume the same graph object with the response.
pending = run2["pending_approval"]
graph2 = run2["graph"]
state2 = {"correlation_id": run2["correlation_id"], "container_id": "MSCU7391045", "party": "maersk-ops"}

resumed = graph2(
    [{"interruptResponse": {"interruptId": pending.id,
                            "response": {"approved": True, "approver": "ops_lead_2",
                                         "note": "waiver agreed under dispute policy"}}}],
    invocation_state=state2,
)

print("status         :", resumed.status)
print("path           :", " -> ".join(n.node_id for n in resumed.execution_order))
print("node executions:", resumed.execution_count)
print("apply_effects  :", str(resumed.results["apply_effects"].result).strip()[:300])
print("\naudit trail from the hook:")
for row in run2["controls"].audit:
    print("  ", row)
print("\npolicy divergences:", run2["controls"].divergences or "none")

### Read the numbers in the last three cells

| Claim | Where you just saw it |
|---|---|
| Resuming does not replay completed nodes | `intake`, `diagnose`, `contract` and `plan` appear once each in the resumed path. The swarm did not run twice, so the resume cost no diagnosis tokens |
| The interrupt carried structured context | `pending_approval.reason` was a dict with exposure, threshold and step list, ready to render in an approval queue |
| Nothing was written before approval | the ledger was unchanged while the run sat interrupted |
| The human response is data, not a string | an approver identity travelled with the decision, which is what an audit needs |

**Production reading of this:** the interrupted result is serialisable, so a real deployment persists it, returns a ticket to the caller, and resumes hours later when the approver acts. `Graph.serialize_state` and `set_session_manager` exist for exactly that, and are the next thing to reach for once approvals leave the notebook.

In [ ]:
# RUN 3. Same case, denied. Denial is a destination: the graph routes to rejected,
# writes the audit row, and touches nothing else.
ledger_before = len(EFFECT_LEDGER)

run3 = run_release(
    "MSCU7391045", "maersk-ops",
    approval_response={"approved": False, "approver": "ops_lead_2",
                       "note": "dispute unresolved, waiver not authorised"},
)
show(run3)
print("\nledger entries before:", ledger_before, "after:", len(EFFECT_LEDGER))
print("rejected payload:", str(run3["result"].results["rejected"].result).strip())

In [ ]:
# RUN 4. The invariant gate, before a single token is spent.
run4 = run_release("MSC7391045", "unknown-broker")     # bad container shape, unauthorised party
show(run4)
print("\ntokens spent:", run4["result"].accumulated_usage)
print("nodes executed:", [n.node_id for n in run4["result"].execution_order])

### The four runs, as a table you can hand to an architect

| Run | Case | Path taken | Model calls | Human | Side effect |
|---|---|---|---|---|---|
| 1 | Equipment fault, no exposure | intake, diagnose, contract, plan, approval, apply_effects, audit | swarm, contract, planner | none, policy auto-approved | release applied |
| 2 | Customs plus demurrage, exposure above threshold | stops after plan | swarm, contract, planner | approval requested | none while pending |
| 2b | Resumed with approval | approval, apply_effects, audit | zero extra diagnosis calls | named approver recorded | release applied once |
| 3 | Same, denied | intake, diagnose, contract, plan, approval, rejected, audit | swarm, contract, planner | named approver recorded | none |
| 4 | Bad container id, unauthorised party | intake, rejected, audit | **zero** | none | none |

Run 4 is the one to point at in a design review. The cheapest possible check ran first and the expensive part of the system never woke up.

## 5. The production controls, and where each one lives

| # | Control | Implementation in this notebook | Failure it prevents |
|---|---|---|---|
| 1 | Invariant gate before spend | `intake` PythonNode plus edge conditions | Paying tokens to process garbage |
| 2 | Read-only fence on exploration | tool set plus `ReadOnlyFence` on `AfterToolCallEvent` | A specialist mutating state mid-diagnosis |
| 3 | Typed boundary | `contract` node, `DiagnosisReport`, `RemediationPlan` | Silent prose parse failures |
| 4 | Coercion layer | `_clip`, `_to_choice`, `_to_bool`, `_null_to_list` with logging | A boundary model that raises on a stray null or a long sentence |
| 5 | Fail closed on money | `_to_money` returns unknown, `approval_required` treats unknown as approval | A missing exposure defaulting to zero and slipping under the threshold |
| 6 | Policy over model opinion | hook recomputes approval, ignoring `model_thinks_approval_needed` | Model waving through its own fee waiver |
| 7 | Human approval | `event.interrupt()` before `approval` | Unapproved financial exposure |
| 8 | Idempotent effects | correlation-scoped ledger key | Double release on a retry |
| 9 | Caps everywhere | `max_node_executions`, `node_timeout`, `execution_timeout`, `max_handoffs`, repetitive handoff detection | Runaway loops and runaway bills |
| 10 | Audit and cost channel | hooks plus `accumulated_usage` | Being unable to answer what happened and what it cost |
| 11 | Nothing raises inside a side-effect node | `apply_effects` returns an error payload | A raise that skips the audit node and leaves no record |

Two more that deserve their own cells: **concurrency** and **telemetry**.

In [ ]:
# CONCURRENCY. This is the cheapest bug to write and the most expensive to find.
# Deterministic nodes only, so this costs nothing and proves the point exactly.

def tiny_graph():
    b = GraphBuilder()
    for nid in ("a", "b", "c"):
        b.add_node(PythonNode(nid, lambda t, s, nid=nid: {"node": nid, "req": s.get("req")}), nid)
    b.add_edge("a", "b")
    b.add_edge("b", "c")
    b.set_entry_point("a")
    b.set_max_node_executions(10)
    return b.build()


shared = tiny_graph()          # the mistake: one graph object, many requests


async def two_requests_on(graph_or_factory, shared_object: bool):
    g1 = graph_or_factory if shared_object else graph_or_factory()
    g2 = graph_or_factory if shared_object else graph_or_factory()
    return await asyncio.gather(
        g1.invoke_async("task", invocation_state={"req": "customer-A"}),
        g2.invoke_async("task", invocation_state={"req": "customer-B"}),
        return_exceptions=True,
    )


print("SHARED graph object, two concurrent requests")
for i, r in enumerate(asyncio.run(two_requests_on(shared, True))):
    if isinstance(r, Exception):
        print(f"  caller {i}: {type(r).__name__}: {str(r)[:80]}")
    else:
        print(f"  caller {i}: executions={r.execution_count} path={[n.node_id for n in r.execution_order]}")

print("\nGRAPH PER REQUEST")
for i, r in enumerate(asyncio.run(two_requests_on(tiny_graph, False))):
    print(f"  caller {i}: executions={r.execution_count} path={[n.node_id for n in r.execution_order]}")

### The concurrency rule

A three-node graph run twice concurrently on one object reports six executions to both callers, with every node appearing twice in each caller's `execution_order`. No exception. No warning. Two customers' runs merged into one bookkeeping object.

| Primitive | Behaviour under concurrent reuse |
|---|---|
| `Agent` | Raises `ConcurrencyException`. The SDK protects you |
| `Graph` | Silently merges state. Nothing protects you |
| `Swarm` | Holds node history and handoff state per instance. Same exposure |

**Rule: build the graph, the swarm and their agents per request.** Construction is cheap, it allocates objects and does not call Bedrock. In the AgentCore entrypoint later in this notebook, the factory call sits inside the handler for exactly this reason.

Sequential reuse of one graph object is fine and resets correctly. It is concurrency that breaks it, which is why this never shows up in notebook testing and always shows up in production.

In [ ]:
# COST AND LATENCY. The Cohort 1 gap was that per-request cost could not be
# reconstructed afterwards. Emit it at the time, per request, with the correlation id.

# Illustrative rates only. Replace with your current Bedrock pricing before quoting anything.
PRICES_PER_1K = {
    "reasoning": {"in": 0.0008, "out": 0.004},
    "cheap": {"in": 0.00006, "out": 0.00024},
}


def cost_report(run: dict, tier: str = "reasoning") -> dict:
    r = run["result"]
    usage = r.accumulated_usage or {}
    tokens_in = usage.get("inputTokens", 0)
    tokens_out = usage.get("outputTokens", 0)
    rate = PRICES_PER_1K[tier]
    est = (tokens_in / 1000) * rate["in"] + (tokens_out / 1000) * rate["out"]
    node_ms = {row["node"]: row["ms"] for row in run["controls"].audit if row.get("ms") is not None}
    report = {
        "correlation_id": run["correlation_id"],
        "status": str(r.status),
        "input_tokens": tokens_in,
        "output_tokens": tokens_out,
        "est_usd": round(est, 5),
        "graph_ms": round(r.execution_time) if r.execution_time else None,
        "node_ms": node_ms,
        "slowest_node": max(node_ms, key=node_ms.get) if node_ms else None,
    }
    jlog("request_cost", **report)
    return report


for label, run in [("run1 auto-approved", run1), ("run3 denied", run3), ("run4 rejected at intake", run4)]:
    rep = cost_report(run)
    print(f"\n{label}")
    print(f"  tokens in/out : {rep['input_tokens']}/{rep['output_tokens']}   est_usd={rep['est_usd']}")
    print(f"  slowest node  : {rep['slowest_node']}   node_ms={rep['node_ms']}")

In [ ]:
# TELEMETRY. Traces for every agent, tool call and node, exported over OTLP.
# Leave this off unless you want spans: it is global process state.

ENABLE_TELEMETRY = False       # set True to emit
TELEMETRY_TO_CONSOLE = True    # console exporter is loud; useful once, then switch to OTLP

if ENABLE_TELEMETRY:
    from strands.telemetry import StrandsTelemetry

    os.environ.setdefault("OTEL_SERVICE_NAME", "pierpoint-release-desk")
    telemetry = StrandsTelemetry()
    if TELEMETRY_TO_CONSOLE:
        telemetry.setup_console_exporter()
    else:
        # Requires OTEL_EXPORTER_OTLP_ENDPOINT, and a collector or ADOT sidecar at the other end.
        telemetry.setup_otlp_exporter()
    telemetry.setup_meter(enable_console_exporter=TELEMETRY_TO_CONSOLE)
    print("telemetry on. Re-run a release to see spans.")
else:
    print("telemetry off. Set ENABLE_TELEMETRY = True to emit spans and metrics.")

# Guardrails, when the account has one configured, attach at the model:
#
# build_model(REASONING_MODEL) -> BedrockModel(
#     model_id=..., region_name=..., temperature=...,
#     guardrail_id="abcd1234", guardrail_version="1",
#     guardrail_trace="enabled", guardrail_redact_input=True,
# )
#
# Guardrails filter content. They do not enforce business invariants, which is
# why every control in the table above is still required.

## 6. Deploying to AgentCore Runtime, from this notebook

AgentCore Runtime is an HTTP contract in front of your container:

| Path | Method | Purpose |
|---|---|---|
| `/invocations` | POST | your entrypoint receives the JSON payload |
| `/ping` | GET | health, returns a status the platform polls |

Port 8080, session identity carried on the `X-Amzn-Bedrock-AgentCore-Runtime-Session-Id` header and surfaced to your code as `RequestContext.session_id`.

```mermaid
flowchart LR
    CALLER["Caller: SDK, API, or console"] --> RT["AgentCore Runtime endpoint"]
    RT --> APP["Your container, port 8080"]
    APP --> EP["entrypoint: builds graph PER REQUEST"]
    EP --> BR["Bedrock models"]
    APP --> LOGS["stdout, structured JSON"]
    LOGS --> CW["CloudWatch log group per agent"]
    APP --> SPANS["OTEL spans"]
    SPANS --> CW
```

Two paths from here, and the local one is the one to run in class:

| Path | Needs | Proves |
|---|---|---|
| Local server, same contract | nothing beyond Bedrock creds | the entrypoint, payload shape, session handling, logging |
| Cloud deploy via the starter toolkit | ECR, CodeBuild, an execution role, `iam:PassRole` | the deployed artefact, IAM wiring, CloudWatch integration |

The next cells write two files. `release_desk_core.py` holds the logic, `release_desk_agent.py` is the thin runtime adapter. That split is the point: the notebook was the design surface, the module is the deployable unit, and the entrypoint is twenty lines that own nothing.

In [ ]:
# Write the deployable core. Same design as above, trimmed to what the runtime needs.
# Tool descriptions are passed explicitly here rather than as docstrings, purely because
# this file is generated from a notebook string. Both forms produce the same tool spec.

CORE_SRC = """
import json
import logging
import os
import re
import sys
import time
import uuid
from typing import Any, AsyncIterator, Callable, Literal

from botocore.config import Config as BotocoreConfig
from pydantic import BaseModel, Field
from strands import Agent, ModelRetryStrategy, tool
from strands.agent.agent_result import AgentResult
from strands.models import BedrockModel
from strands.multiagent import GraphBuilder, Swarm
from strands.multiagent.graph import GraphState
from strands.hooks import AfterNodeCallEvent, BeforeNodeCallEvent, HookProvider, HookRegistry
from typing import Annotated
from pydantic import BeforeValidator

def _null_to_list(value):
    # Models emit null for empty lists. default_factory only fires when the key is absent.
    return [] if value is None else value

def _clip(limit):
    def clip(value):
        if isinstance(value, str) and len(value) > limit:
            return value[:limit]
        return value
    return clip

def _to_choice(allowed, fallback):
    def choose(value):
        if isinstance(value, str):
            candidate = value.strip().lower().replace(' ', '_').replace('-', '_')
            if candidate in allowed:
                return candidate
            for option in allowed:
                if option in candidate:
                    return option
        return fallback
    return choose

def _to_bool(value):
    # Unreadable means blocking: wrong in the safe direction.
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        text = value.strip().lower()
        if text in {'true', 'yes', 'y', '1', 'blocking'}:
            return True
        if text in {'false', 'no', 'n', '0', 'clear', 'not_blocking'}:
            return False
    return True

def _to_money(value):
    # None means unknown. Unknown forces approval; it must never become a zero.
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return float(value) if value >= 0 else None
    if isinstance(value, str):
        m = re.search('-?[0-9]+(?:[.][0-9]+)?', value.replace(',', ''))
        if m:
            parsed = float(m.group())
            return parsed if parsed >= 0 else None
    return None

def approval_required(exposure_usd, threshold_usd):
    # Fail closed: unknown exposure always needs a human.
    return exposure_usd is None or exposure_usd > threshold_usd

AREAS = ('customs', 'billing', 'damage', 'equipment')
ACTIONS = ('clear_customs_hold', 'waive_demurrage', 'post_charge',
           'dispatch_equipment_repair', 'request_survey', 'notify_party')

REGION = os.environ.get('AWS_REGION', 'us-east-1')
REASONING_MODEL = os.environ.get('REASONING_MODEL', 'us.anthropic.claude-haiku-4-5-20251001-v1:0')
CHEAP_MODEL = os.environ.get('CHEAP_MODEL', 'amazon.nova-lite-v1:0')
APPROVAL_THRESHOLD_USD = float(os.environ.get('APPROVAL_THRESHOLD_USD', '500'))
AUTHORIZED_PARTIES = {'maersk-ops', 'cma-cgm-ops', 'pierpoint-internal'}
CONTAINER_PATTERN = re.compile('^[A-Z]{4}[0-9]{7}$')

logging.basicConfig(level=logging.INFO, format='%(message)s', stream=sys.stdout, force=True)
_log = logging.getLogger('release_desk')

def jlog(event, **fields):
    _log.info(json.dumps({'event': event, **fields}, default=str))

BOTO_CFG = BotocoreConfig(retries={'max_attempts': 5, 'mode': 'adaptive'}, read_timeout=90, connect_timeout=10)
RETRY = ModelRetryStrategy(max_attempts=5, initial_delay=4, max_delay=60)

def build_model(model_id, temperature=0.2, max_tokens=900):
    return BedrockModel(model_id=model_id, region_name=REGION, temperature=temperature,
                        max_tokens=max_tokens, boto_client_config=BOTO_CFG)

CUSTOMS = {'MSCU7391045': {'hold': 'documentary', 'reason': 'invoice value mismatch', 'cleared': False},
           'CAIU9083321': {'hold': 'none', 'reason': '', 'cleared': True}}
BILLING = {'MSCU7391045': {'demurrage_days': 6, 'accrued_usd': 1800.0, 'dispute_open': True},
           'CAIU9083321': {'demurrage_days': 0, 'accrued_usd': 0.0, 'dispute_open': False}}
SURVEY = {'MSCU7391045': {'survey_done': False, 'damage': 'none reported'},
          'CAIU9083321': {'survey_done': True, 'damage': 'none'}}
EQUIPMENT = {'MSCU7391045': {'fault': 'none', 'reefer_required': False},
             'CAIU9083321': {'fault': 'reefer plug bay R04 no power', 'reefer_required': True}}

def _lookup(table, container_id, label):
    rec = table.get(str(container_id).strip().upper())
    if rec is None:
        return 'No ' + label + ' record exists for container ' + str(container_id) + '.'
    return json.dumps(rec)

@tool(name='customs_status', description='Read the customs hold status for one container. Argument container_id is an ISO 6346 number, four letters then seven digits.')
def customs_status(container_id: str) -> str:
    jlog('tool_call', tool='customs_status', container_id=container_id)
    return _lookup(CUSTOMS, container_id, 'customs')

@tool(name='billing_status', description='Read demurrage, accrued charges and dispute state for one container. Argument container_id is an ISO 6346 number.')
def billing_status(container_id: str) -> str:
    jlog('tool_call', tool='billing_status', container_id=container_id)
    return _lookup(BILLING, container_id, 'billing')

@tool(name='damage_survey', description='Read the damage survey record for one container. Argument container_id is an ISO 6346 number.')
def damage_survey(container_id: str) -> str:
    jlog('tool_call', tool='damage_survey', container_id=container_id)
    return _lookup(SURVEY, container_id, 'survey')

@tool(name='equipment_status', description='Read equipment and reefer power faults for one container. Argument container_id is an ISO 6346 number.')
def equipment_status(container_id: str) -> str:
    jlog('tool_call', tool='equipment_status', container_id=container_id)
    return _lookup(EQUIPMENT, container_id, 'equipment')

READ_ONLY_TOOLS = [customs_status, billing_status, damage_survey, equipment_status]

HANDOFF_BRIEF = ('You are one of four PierPoint release-desk specialists: customs_specialist, '
                 'billing_specialist, damage_specialist, equipment_specialist. Check your own area '
                 'with your tool first, report in two sentences, and call handoff_to_agent when the '
                 'finding points at another area. Stop when nothing else needs checking. You cannot '
                 'change anything.')
SPECIALISTS = {'customs_specialist': 'Customs holds and documentary discrepancies.',
               'billing_specialist': 'Demurrage, accrued charges, billing disputes.',
               'damage_specialist': 'Damage surveys and condition disputes.',
               'equipment_specialist': 'Reefer power, plug bays, mechanical faults.'}

class Finding(BaseModel):
    area: Annotated[Literal['customs', 'billing', 'damage', 'equipment'],
                    BeforeValidator(_to_choice(AREAS, 'customs'))] = 'customs'
    blocking: Annotated[bool, BeforeValidator(_to_bool)] = True
    detail: Annotated[str, BeforeValidator(_clip(300))] = ''

class DiagnosisReport(BaseModel):
    container_id: str = ''
    findings: Annotated[list[Finding], BeforeValidator(_null_to_list)] = Field(default_factory=list)
    primary_blocker: Annotated[Literal['customs', 'billing', 'damage', 'equipment', 'none'],
                               BeforeValidator(_to_choice(AREAS + ('none',), 'none'))] = 'none'
    unresolved: Annotated[list[str], BeforeValidator(_null_to_list)] = Field(default_factory=list)

CONTRACT_BRIEF = ('Convert a release-desk investigation transcript into the required structure. '
                  'One finding per area actually checked. Set blocking=true only where the '
                  'transcript says the container cannot move. Never invent an area. Use an empty '
                  'list where a list is empty, never null.')

class PlanStep(BaseModel):
    action: Annotated[Literal['clear_customs_hold', 'waive_demurrage', 'post_charge',
                              'dispatch_equipment_repair', 'request_survey', 'notify_party'],
                      BeforeValidator(_to_choice(ACTIONS, 'notify_party'))] = 'notify_party'
    target: str = ''
    rationale: Annotated[str, BeforeValidator(_clip(200))] = ''

class RemediationPlan(BaseModel):
    container_id: str = ''
    steps: Annotated[list[PlanStep], BeforeValidator(_null_to_list)] = Field(default_factory=list)
    financial_exposure_usd: Annotated[float | None, BeforeValidator(_to_money)] = None
    model_thinks_approval_needed: Annotated[bool, BeforeValidator(_to_bool)] = False
    customer_message: Annotated[str, BeforeValidator(_clip(600))] = ''

PLANNER_BRIEF = ('You are the PierPoint release desk planner. One step per blocking finding, none for '
                 'clear areas. financial_exposure_usd is the total value PierPoint gives up or defers, '
                 'zero if none. Never admit liability. Under 120 words.')

EFFECT_LEDGER = {}

class PythonNode:
    def __init__(self, name, fn):
        self.name = name
        self._fn = fn
    @staticmethod
    def _wrap(payload):
        return AgentResult(stop_reason='end_turn',
                           message={'role': 'assistant', 'content': [{'text': json.dumps(payload)}]},
                           metrics=None, state={})
    async def invoke_async(self, prompt=None, **kwargs):
        text = ''.join(b.get('text', '') for b in (prompt or []) if isinstance(b, dict))
        payload = self._fn(text, kwargs.get('invocation_state') or {})
        jlog('python_node', node=self.name, payload=payload)
        return self._wrap(payload)
    def __call__(self, prompt=None, **kwargs):
        import asyncio
        return asyncio.run(self.invoke_async(prompt, **kwargs))
    async def stream_async(self, prompt=None, **kwargs) -> AsyncIterator[Any]:
        yield {'result': await self.invoke_async(prompt, **kwargs)}

def node_json(holder, node_id):
    nr = holder.results.get(node_id)
    if nr is None:
        return {}
    raw = str(nr.result).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        s, e = raw.find('{'), raw.rfind('}')
        if s >= 0 and e > s:
            try:
                return json.loads(raw[s:e + 1])
            except json.JSONDecodeError:
                pass
        jlog('node_parse_failed', node=node_id, raw=raw[:200])
        return {'_parse_failed': True}

def intake_fn(_text, state):
    cid = str(state.get('container_id', '')).strip().upper()
    party = str(state.get('party', '')).strip().lower()
    problems = []
    if not CONTAINER_PATTERN.match(cid):
        problems.append('container_id is not ISO 6346 shape')
    if party not in AUTHORIZED_PARTIES:
        problems.append('party is not authorized to request a release')
    return {'ok': not problems, 'problems': problems, 'container_id': cid, 'party': party}

def approval_fn(_text, state):
    decision = state.get('approval_outcome')
    if decision is None:
        return {'approved': False, 'by': 'missing', 'reason': 'no approval decision present'}
    return decision

def apply_effects_fn(_text, state):
    cid, corr = state.get('container_id'), state.get('correlation_id')
    if not cid or not corr:
        jlog('apply_effects_missing_context', container_id=cid, correlation_id=corr)
        return {'status': 'error', 'reason': 'missing container_id or correlation_id', 'effects': []}
    plan = state.get('plan') or {}
    key = corr + ':release:' + cid
    if key in EFFECT_LEDGER:
        return {'status': 'already_applied', 'key': key, 'effects': EFFECT_LEDGER[key]['effects']}
    effects = [{'action': s.get('action'), 'target': s.get('target'), 'result': 'applied'}
               for s in plan.get('steps') or []]
    effects.append({'action': 'release_container', 'target': cid, 'result': 'applied'})
    EFFECT_LEDGER[key] = {'effects': effects}
    return {'status': 'applied', 'key': key, 'effects': effects}

def rejected_fn(_text, state):
    return {'outcome': 'not_released', 'container_id': state.get('container_id'),
            'reason': state.get('rejection_reason', 'approval not granted'), 'customer_notified': True}

def audit_fn(_text, state):
    return {'audit_written': True, 'correlation_id': state.get('correlation_id'),
            'container_id': state.get('container_id')}

class ReleaseControls(HookProvider):
    def __init__(self, threshold_usd=APPROVAL_THRESHOLD_USD):
        self.threshold = threshold_usd
        self.audit = []
        self._started = {}
    def register_hooks(self, registry: HookRegistry, **kwargs) -> None:
        registry.add_callback(BeforeNodeCallEvent, self.before_node)
        registry.add_callback(AfterNodeCallEvent, self.after_node)
    def before_node(self, event: BeforeNodeCallEvent) -> None:
        self._started[event.node_id] = time.time()
        if event.node_id == 'rejected' and 'rejection_reason' not in event.invocation_state:
            problems = node_json(event.source.state, 'intake').get('problems') or []
            if problems:
                event.invocation_state['rejection_reason'] = '; '.join(problems)
        if event.node_id != 'approval':
            return
        plan = node_json(event.source.state, 'plan')
        event.invocation_state['plan'] = plan
        exposure = _to_money(plan.get('financial_exposure_usd'))
        if not approval_required(exposure, self.threshold):
            event.invocation_state['approval_outcome'] = {'approved': True, 'by': 'policy',
                                                         'exposure_usd': exposure}
            return
        decision = event.interrupt('release_approval',
                                   reason={'container_id': plan.get('container_id'),
                                           'exposure_usd': exposure,
                                           'exposure_known': exposure is not None,
                                           'threshold_usd': self.threshold})
        event.invocation_state['approval_outcome'] = {'approved': bool(decision.get('approved')),
                                                      'by': decision.get('approver', 'unknown'),
                                                      'exposure_usd': exposure}
        if not decision.get('approved'):
            event.invocation_state['rejection_reason'] = decision.get('note', 'approval denied')
    def after_node(self, event: AfterNodeCallEvent) -> None:
        started = self._started.pop(event.node_id, None)
        row = {'node': event.node_id, 'ms': round((time.time() - started) * 1000) if started else None}
        self.audit.append(row)
        jlog('node_complete', **row)

def build_diagnostic_swarm():
    agents = [Agent(model=build_model(REASONING_MODEL, 0.2, 700), name=name,
                    system_prompt=HANDOFF_BRIEF + ' Your area: ' + area,
                    tools=READ_ONLY_TOOLS, retry_strategy=RETRY, callback_handler=None)
              for name, area in SPECIALISTS.items()]
    return Swarm(agents, entry_point=agents[0], max_handoffs=6, max_iterations=8,
                 execution_timeout=240.0, node_timeout=60.0,
                 repetitive_handoff_detection_window=3, repetitive_handoff_min_unique_agents=2)

def build_contract_agent(model_id=CHEAP_MODEL):
    return Agent(model=build_model(model_id, 0.0, 900), name='contract',
                 system_prompt=CONTRACT_BRIEF, structured_output_model=DiagnosisReport,
                 retry_strategy=RETRY, callback_handler=None)

def build_planner():
    return Agent(model=build_model(REASONING_MODEL, 0.2, 1200), name='planner',
                 system_prompt=PLANNER_BRIEF, structured_output_model=RemediationPlan,
                 retry_strategy=RETRY, callback_handler=None)

def _approved(state: GraphState) -> bool:
    return bool(node_json(state, 'approval').get('approved'))

def _denied(state: GraphState) -> bool:
    return 'approval' in state.results and not node_json(state, 'approval').get('approved')

def _intake_ok(state: GraphState) -> bool:
    return bool(node_json(state, 'intake').get('ok'))

def _intake_failed(state: GraphState) -> bool:
    return 'intake' in state.results and not node_json(state, 'intake').get('ok')

def build_release_graph(controls, swarm):
    b = GraphBuilder()
    b.add_node(PythonNode('intake', intake_fn), 'intake')
    b.add_node(swarm, 'diagnose')
    b.add_node(build_contract_agent(), 'contract')
    b.add_node(build_planner(), 'plan')
    b.add_node(PythonNode('approval', approval_fn), 'approval')
    b.add_node(PythonNode('apply_effects', apply_effects_fn), 'apply_effects')
    b.add_node(PythonNode('rejected', rejected_fn), 'rejected')
    b.add_node(PythonNode('audit', audit_fn), 'audit')
    b.add_edge('intake', 'diagnose', condition=_intake_ok)
    b.add_edge('intake', 'rejected', condition=_intake_failed)
    b.add_edge('diagnose', 'contract')
    b.add_edge('contract', 'plan')
    b.add_edge('plan', 'approval')
    b.add_edge('approval', 'apply_effects', condition=_approved)
    b.add_edge('approval', 'rejected', condition=_denied)
    b.add_edge('apply_effects', 'audit')
    b.add_edge('rejected', 'audit')
    b.set_entry_point('intake')
    b.set_hook_providers([controls])
    b.set_max_node_executions(16)
    b.set_node_timeout(120.0)
    b.set_execution_timeout(600.0)
    return b.build()

def handle_release(container_id, party, correlation_id=None, approval=None, session_id=None):
    correlation_id = correlation_id or ('req-' + uuid.uuid4().hex[:8])
    controls = ReleaseControls()
    graph = build_release_graph(controls, build_diagnostic_swarm())
    state = {'correlation_id': correlation_id, 'container_id': container_id, 'party': party}
    jlog('request_start', correlation_id=correlation_id, session_id=session_id,
         container_id=container_id, party=party)
    task = ('Release request for container ' + str(container_id) + ' from ' + str(party) +
            '. Diagnose every blocker, then plan the remediation.')
    result = graph(task, invocation_state=state)
    if result.status.name == 'INTERRUPTED' and result.interrupts:
        pending = result.interrupts[0]
        if approval is None:
            jlog('approval_required', correlation_id=correlation_id, interrupt_id=pending.id)
            return {'status': 'APPROVAL_REQUIRED', 'correlation_id': correlation_id,
                    'interrupt_id': pending.id, 'reason': pending.reason}
        result = graph([{'interruptResponse': {'interruptId': pending.id, 'response': approval}}],
                       invocation_state=state)
    payload = {'status': str(result.status), 'correlation_id': correlation_id,
               'path': [n.node_id for n in result.execution_order],
               'usage': result.accumulated_usage,
               'outcome': node_json(result, 'apply_effects') or node_json(result, 'rejected')}
    jlog('request_complete', **payload)
    return payload
"""

Path("release_desk_core.py").write_text(CORE_SRC.strip() + "\n", encoding="utf-8")
print("wrote release_desk_core.py", len(CORE_SRC.splitlines()), "lines")

import py_compile
py_compile.compile("release_desk_core.py", doraise=True)
print("syntax ok")

In [ ]:
# The runtime adapter. Twenty lines that own no business logic.
AGENT_SRC = """
import os

from bedrock_agentcore.runtime import BedrockAgentCoreApp, RequestContext

from release_desk_core import handle_release, jlog

app = BedrockAgentCoreApp()

@app.entrypoint
def invoke(payload: dict, context: RequestContext = None):
    session_id = getattr(context, 'session_id', None)
    container_id = payload.get('container_id')
    party = payload.get('party', 'pierpoint-internal')
    approval = payload.get('approval')
    correlation_id = payload.get('correlation_id')
    if not container_id:
        return {'status': 'BAD_REQUEST', 'error': 'container_id is required'}
    try:
        return handle_release(container_id=container_id, party=party,
                              correlation_id=correlation_id, approval=approval,
                              session_id=session_id)
    except Exception as exc:
        jlog('unhandled_error', error=type(exc).__name__, detail=str(exc)[:400])
        return {'status': 'ERROR', 'error': type(exc).__name__}

if __name__ == '__main__':
    app.run(port=int(os.environ.get('PORT', '8080')))
"""

# Pin to the versions this kernel actually ran, so the container matches your tests.
import importlib.metadata as _md

REQS = "\n".join([
    f"strands-agents=={_md.version('strands-agents')}",
    f"bedrock-agentcore=={_md.version('bedrock-agentcore')}",
    "pydantic>=2.7",
    "boto3>=1.35",
]) + "\n"

Path("release_desk_agent.py").write_text(AGENT_SRC.strip() + "\n", encoding="utf-8")
Path("requirements.txt").write_text(REQS, encoding="utf-8")

import py_compile
py_compile.compile("release_desk_agent.py", doraise=True)
print("wrote release_desk_agent.py and requirements.txt, syntax ok")

# Prove the module and the notebook agree before deploying anything.
import importlib, release_desk_core
importlib.reload(release_desk_core)
print("core imports cleanly, threshold =", release_desk_core.APPROVAL_THRESHOLD_USD)

In [ ]:
# LOCAL RUN. Same HTTP contract as the cloud, no Docker, no IAM beyond Bedrock.
import requests


def free_port(start: int = 8080, tries: int = 20) -> int:
    """Bind-test to find a port. Never shell out to lsof: a stray kill on your own PID
    takes the Jupyter kernel with it."""
    for port in range(start, start + tries):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                sock.bind(("127.0.0.1", port))
                return port
            except OSError:
                continue
    raise RuntimeError("no free port in range")


PORT = free_port()
env = {**os.environ, "PORT": str(PORT), "AWS_REGION": REGION}
proc = subprocess.Popen(
    [sys.executable, "release_desk_agent.py"],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
print(f"started pid={proc.pid} on port {PORT}")

base = f"http://127.0.0.1:{PORT}"
ready = False
for _ in range(40):
    if proc.poll() is not None:
        print("process exited early:\n", proc.stdout.read()[-2000:])
        break
    try:
        ping = requests.get(f"{base}/ping", timeout=1)
        if ping.status_code == 200:
            ready = True
            print("ping:", ping.status_code, ping.text.strip())
            break
    except requests.RequestException:
        time.sleep(0.5)
print("ping ok:", ready)

In [ ]:
# Invoke the local runtime exactly as AgentCore would.
if ready:
    session_header = {"X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": f"sess-{uuid.uuid4().hex[:12]}"}

    print("--- case with no financial exposure ---")
    r = requests.post(f"{base}/invocations",
                      json={"container_id": "CAIU9083321", "party": "cma-cgm-ops"},
                      headers=session_header, timeout=300)
    print(json.dumps(r.json(), indent=2)[:900])

    print("\n--- case that needs approval ---")
    r2 = requests.post(f"{base}/invocations",
                       json={"container_id": "MSCU7391045", "party": "maersk-ops"},
                       headers=session_header, timeout=300)
    pending = r2.json()
    print(json.dumps(pending, indent=2)[:900])

    print("\n--- invariant gate, zero tokens ---")
    r3 = requests.post(f"{base}/invocations",
                       json={"container_id": "MSC7391045", "party": "unknown-broker"},
                       headers=session_header, timeout=60)
    print(json.dumps(r3.json(), indent=2)[:600])
else:
    print("server not ready, skipping invocations")

In [ ]:
# Always stop the local server. Terminate the child by handle, never by port scan.
if proc.poll() is None:
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proc.kill()
print("local server stopped, exit code:", proc.poll())

out = proc.stdout.read() if proc.stdout else ""
lines = [ln for ln in out.splitlines() if ln.strip().startswith("{")]
print(f"\nstructured log lines captured: {len(lines)} (this is what CloudWatch would receive)")
for ln in lines[:6]:
    print("  ", ln[:160])

### What the local run established

| Checked | Result |
|---|---|
| `/ping` and `/invocations` contract | the runtime adapter answers both, so the container is deployable |
| Payload shape | plain JSON in, plain JSON out. No Strands types cross the boundary |
| Session identity | header reaches the entrypoint as `RequestContext.session_id` and lands in the logs |
| Approval over HTTP | the interrupt became an `APPROVAL_REQUIRED` response with an `interrupt_id`, which is how an async approval queue is fed |
| Logs | every line is JSON on stdout, which is precisely what AgentCore ships to CloudWatch |

Two operational habits worth keeping from this cell: find ports by bind-testing, and stop child processes by handle. Killing by port scan is how a notebook takes down its own kernel.

**One honest gap.** The interrupt state lives in the graph object in memory, so this process can resume it and a fresh one cannot. Durable approvals need `Graph.serialize_state` or a session manager behind the endpoint. That is the next build, and it is the reason `APPROVAL_REQUIRED` returns an id rather than pretending to block.

In [ ]:
# CI GATE. Every assertion below runs with no AWS credentials and zero tokens.
# This is the part of an agentic system that belongs in a normal test pipeline.

# 1. invariant gate
assert intake_fn("", {"container_id": "MSCU7391045", "party": "maersk-ops"})["ok"] is True
assert intake_fn("", {"container_id": "MSC7391045", "party": "maersk-ops"})["ok"] is False
assert intake_fn("", {"container_id": "MSCU7391045", "party": "not-a-customer"})["ok"] is False

# 2. idempotency: the second apply must not write a second effect
before = len(EFFECT_LEDGER)
ci_state = {"container_id": "TCLU1234567", "correlation_id": "ci-run-1",
            "plan": {"steps": [], "financial_exposure_usd": 0.0}}
first = apply_effects_fn("", ci_state)
second = apply_effects_fn("", ci_state)
assert first["status"] == "applied" and second["status"] == "already_applied"
assert len(EFFECT_LEDGER) == before + 1


# 3. edge conditions, against a minimal stand-in for graph state
class _NR:
    def __init__(self, text: str) -> None:
        self.result = text


class _Holder:
    def __init__(self, mapping: dict) -> None:
        self.results = mapping


approved_state = _Holder({"approval": _NR(json.dumps({"approved": True}))})
denied_state = _Holder({"approval": _NR(json.dumps({"approved": False}))})
assert request_approved(approved_state) and not request_denied(approved_state)
assert request_denied(denied_state) and not request_approved(denied_state)

# 4. a parse failure must be flagged, never silently treated as a denial
assert node_json(_Holder({"plan": _NR("the plan is to release it")}), "plan") == {"_parse_failed": True}

# 5. the read-only fence catches a write tool by name
fence = ReadOnlyFence(READ_ONLY_TOOL_NAMES)
assert "release_container" not in fence.allowed

# 6. the coercion layer, one assertion per boundary failure mode
assert DiagnosisReport(**{"unresolved": None}).unresolved == []                      # null list
assert DiagnosisReport(**{"findings": None}).findings == []                          # null list
assert len(Finding(**{"detail": "x" * 400}).detail) == 300                           # over length
assert DiagnosisReport(**{"primary_blocker": "Customs Hold"}).primary_blocker == "customs"
assert Finding(**{"blocking": "yes"}).blocking is True                               # string bool
assert Finding(**{"blocking": "nonsense"}).blocking is True                          # unreadable fails closed
assert RemediationPlan(**{"financial_exposure_usd": "about 1,800 USD"}).financial_exposure_usd == 1800.0
assert RemediationPlan(**{}).financial_exposure_usd is None                          # unknown, never zero
assert RemediationPlan(**{"financial_exposure_usd": -50}).financial_exposure_usd is None

# 7. the money rule itself: unknown must never auto-approve
assert approval_required(None, 500.0) is True
assert approval_required(1800.0, 500.0) is True
assert approval_required(120.0, 500.0) is False
assert approval_required(500.0, 500.0) is False

print("CI gate passed: 7 groups, 0 tokens, 0 AWS calls")

In [ ]:
# CLOUD DEPLOY. Off by default: it creates billable resources.
DEPLOY_TO_CLOUD = False
AGENT_NAME = f"pierpoint_release_desk_{os.environ.get('USER', 'lab')}".replace("-", "_")[:48]
EXECUTION_ROLE = os.environ.get("AGENTCORE_EXECUTION_ROLE_ARN")   # ask your account admin

if DEPLOY_TO_CLOUD:
    from bedrock_agentcore_starter_toolkit import Runtime

    runtime = Runtime()
    cfg = runtime.configure(
        entrypoint="release_desk_agent.py",
        agent_name=AGENT_NAME,
        requirements_file="requirements.txt",
        region=REGION,
        execution_role=EXECUTION_ROLE,            # None means the toolkit tries to create one
        auto_create_execution_role=EXECUTION_ROLE is None,
        auto_create_ecr=True,
        protocol="HTTP",
        non_interactive=True,
    )
    print("configured:", cfg)

    launched = runtime.launch()      # builds in CodeBuild, no local Docker needed
    print("launched:", launched)

    for _ in range(40):
        status = runtime.status()
        text = str(status)
        print("status:", text[:200])
        if "READY" in text or "FAILED" in text:
            break
        time.sleep(15)
    # The log group name is in this status object. Read it from here rather than
    # assembling the path by hand.
else:
    print("DEPLOY_TO_CLOUD is False. Set it True once you have an execution role ARN.")
    print("agent name would be:", AGENT_NAME)

In [ ]:
# Invoke the deployed endpoint, then tear it down. Both guarded by the same flag.
if DEPLOY_TO_CLOUD:
    response = runtime.invoke(
        {"container_id": "CAIU9083321", "party": "cma-cgm-ops"},
        session_id=f"sess-{uuid.uuid4().hex[:12]}",
    )
    print(json.dumps(response, indent=2, default=str)[:1200])

    print("\n--- teardown preview ---")
    print(runtime.destroy(dry_run=True))
    # runtime.destroy(delete_ecr_repo=True)   # uncomment to actually remove everything
else:
    print("skipped")

### Deploying this in a shared training account

A thirty-person cohort on one AWS account hits the same three walls every time.

| Wall | Symptom | Fix before the session |
|---|---|---|
| `iam:PassRole` and `iam:CreateRole` | `configure` fails or `launch` cannot attach a role. `AmazonBedrockFullAccess` does not grant either | Pre-create one execution role, hand out the ARN, set `auto_create_execution_role=False` |
| Name collisions | second learner overwrites the first agent | Suffix `agent_name` per learner, as the cell above does with `USER` |
| Nothing gets deleted | ECR images and runtimes accrue after the session | Run `destroy(dry_run=True)` in class, then the real `destroy` |

Execution role needs, at minimum: `bedrock:InvokeModel` and `bedrock:InvokeModelWithResponseStream` on the inference profile and model ARNs, ECR pull, and CloudWatch Logs write. `bedrock:Converse` is not a valid IAM action: `InvokeModel` covers Converse calls.

## 7. Observability: what to look at, and where it is

AgentCore Runtime sends agent telemetry to one log group per agent. Since the July 2026 change, traces, prompts, structured logs and stdout all land in the same place instead of being split with spans in `aws/spans`.

**Log group:** `/aws/bedrock-agentcore/runtimes/<agent_id>-<endpoint_name>`, stream prefix `[runtime-logs]`. Read the exact value from `runtime.status()` rather than assembling it by hand.

**One-time account setup:** enable CloudWatch Transaction Search, otherwise the GenAI Observability page shows metrics but no traces. Observability is automatic for agents hosted on AgentCore Runtime; anything hosted elsewhere needs its own log group plus the ADOT environment variables.

### Where each signal in this notebook ends up

| Signal | Produced by | Lands in | Question it answers |
|---|---|---|---|
| `jlog` JSON lines | your code, stdout | agent log group | what happened in request X |
| Node timings | `AfterNodeCallEvent` hook | your log lines, then a metric | which node is slow |
| Approval events | `GraphResult.interrupts` | your approval store | how long approvals take, how often they are denied |
| Token usage | `result.accumulated_usage` | `request_cost` log line | cost per request, per customer |
| Spans per agent, tool and node | `StrandsTelemetry` plus OTLP | GenAI Observability, Transaction Search | where latency went inside one invocation |
| Runtime metrics: invocations, sessions, errors, CPU, memory | AgentCore | CloudWatch metrics | is the service healthy |
| Fence violations | `ReadOnlyFence` | log line, and a test assertion | did anything try to write during diagnosis |

### The seven alarms worth having

| Alarm | Signal | Why it matters here |
|---|---|---|
| Cap-hit rate | `status == FAILED` with `execution_count == max_node_executions` | a loop is burning money and returning nothing |
| Interrupt backlog age | oldest pending approval | I2 turns into a queue, and queues rot |
| Swarm hop p95 | `len(node_history)` | at the cap, the cap is your diagnosis |
| Tool error rate per tool | `AfterToolCallEvent.exception` | one broken lookup makes the whole diagnosis confident and wrong |
| Idempotency collisions | `status == already_applied` | retries are firing more than you think |
| Policy divergence rate | `ReleaseControls.divergences` | the planner's judgement is drifting from policy |
| Cost per request p95 | `accumulated_usage` | the number your sponsor asks about |

### How to check it, in order

1. `runtime.status()` for the log group and endpoint state.
2. CloudWatch Logs Insights on that group, filter by `correlation_id` to replay one request end to end.
3. GenAI Observability for the trace of a single invocation, node by node.
4. Your own `request_cost` lines for economics, since token cost is not reconstructable after the fact in a shared account.

Sources: AWS `what's new` on unified AgentCore observability (July 2026), AgentCore runtime observability and troubleshooting guides.

## 8. Summary

**What was built.** One release-desk system: a graph of eight nodes with a four-agent swarm nested inside one of them, a typed contract node between exploration and planning, eleven production controls, an idempotent side-effect node, an HTTP runtime adapter, and a local deployment of the same code that would run in AgentCore.

**The decision, restated.**

| | Graph | Swarm |
|---|---|---|
| Owns | invariants, sequencing, approval gates, side effects, provenance | bounded exploration where the next specialist is unknown |
| Decides the next step | your Python, on an edge | the agents, at runtime |
| Fails by | condition sprawl that becomes an unreadable rules engine | ping-pong, weak provenance, cost that moves with the caps |
| In this system | six of seven stages | one stage, read-only, capped at six hops |

**Rules to carry into the next project.**

1. If you can name the next step at design time, an LLM must not be the thing that picks it.
2. If a step writes, it does not get to decide whether it runs.
3. Prose never crosses a node boundary. Types do.
4. Let the model propose, let code decide, and log the disagreement as a metric.
5. Coerce where the model is sloppy, fail closed where money or safety depends on the field. Unknown must never look like zero.
6. Build the graph per request. `Agent` raises on concurrent reuse, `Graph` does not.
7. `cancel_node` aborts a run. A denied approval is a destination, so it gets an edge.
8. Every side effect gets an idempotency key before it gets a retry policy.
9. A boundary model should be unable to raise. Every field gets a fallback, every fallback gets a log line, and the log line is a metric.

**What this notebook deliberately does not do.**

- No durable approvals. Interrupt state is in memory, so only this process can resume it. `Graph.serialize_state` and session managers are the fix.
- No memory or retrieval. The specialists know nothing beyond the current request, which is the next session's topic.
- No evaluation harness. There is no measurement here of whether the diagnosis was correct, only that the system behaved lawfully.
- Prices in the cost cell are illustrative. Replace them before quoting a number to anyone.

**Three things to try on your own use case.**

1. Take one workflow you own and fill the stage table from section 1. Count the rows that are genuinely not enumerable. Most teams find zero, which means they need a graph and no swarm.
2. Move one invariant out of a prompt and onto an edge. Then write the unit test that would have caught the old version.
3. Add a second approval threshold with a different approver group, and see whether your gate is still one hook or has quietly become a rules engine.